# VT-Guard Buoc 6 - VT-Seq tren Colab (moc 2: OOF chuoi ablation; moc 3: fit cuoi + du doan test MOT lan)

Notebook CHI goi ham trong `src/models/vtseq.py` (khong chep code mo hinh). Quy tac (docs/STEP6-BRIEF.md muc 7):
- Code: `git clone` repo theo **COMMIT co dinh**, kiem `git rev-parse HEAD`.
- Du lieu: CHI `colab_inputs.zip` (tao tren host bang `python src/models/vtseq_aux.py`) gom
  `data/processed/splits/*.jsonl` + `assignment.json`, `vocab.json`, `data/processed/step6/aux.jsonl`. Khong log tho.
- `vtseq.load_data` tu kiem sha256 splits + vocab (so `reports/dataset_stats.json`) va aux (so `reports/step6/aux_manifest.json`); lech -> dung.
- Moc 2 chi tao OOF tren Train + log fold. KHONG cham test o day; cham luon bang `src/eval/evaluate.py` tren host.
- Dau ra moc 2: `outputs/` (oof, logs). Moc 3: `outputs3/` (ckpt, raw, reports/preds, early, sel, oof/logs cua (c)/(c')) -> zip len Drive.
- Chay lai: dat RUN_M2 / RUN_M3 o muc 1 (moc 2 da xong o commit 690103c, OOF da nam trong repo `reports/step6/m2`).

## 1. Tham so

In [ ]:
REPO_URL = "https://github.com/kaitokidc500/VT-Guard-2.git"
COMMIT = ""                      # BAT BUOC: hash commit da push (dien truoc khi chay)
DRIVE_DIR = "/content/drive/MyDrive/vtguard"   # chua colab_inputs.zip; ket qua ghi vao day
INPUT_ZIP = f"{DRIVE_DIR}/colab_inputs.zip"
CONFIGS = ["M1", "M2", "M3", "M4", "M4b"]    # chuoi ablation kien truc (M0 = B3, dung lai Buoc 5)
SEEDS = [1337, 7, 42, 2024, 99]              # brief 4.4; neu qua lau: 3 seed cho M1-M3 (ghi ro)
DEVICE = "cuda"                              # "cpu" neu khong co GPU
RUN_M2 = False                               # moc 2 (OOF) da chay xong; OOF + log nam trong repo reports/step6/m2
RUN_M3 = True                                # moc 3: fit cuoi + du doan test MOT lan
assert len(COMMIT) == 40, "Dien COMMIT (40 ky tu) truoc khi chay"

## 2. Clone repo theo commit

In [ ]:
import os, subprocess
REPO = "/content/VT-Guard"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, (head, COMMIT)
print("HEAD", head)

## 3. Dau vao: chi splits + vocab + aux

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import zipfile
ALLOWED = {f"data/processed/splits/{s}.jsonl" for s in ("train", "t_clean", "t_hard", "t_novel")} | {
    "data/processed/splits/assignment.json", "data/processed/vocab.json", "data/processed/step6/aux.jsonl"}
with zipfile.ZipFile(INPUT_ZIP) as z:
    names = set(z.namelist())
    assert names == ALLOWED, f"zip chua file ngoai danh sach: {sorted(names ^ ALLOWED)}"
    z.extractall(REPO)
print(sorted(names))

## 4. Moi truong + nap du lieu (kiem checksum) + unit test

In [ ]:
import sys, json, time
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # tat dinh tren GPU
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src", "models"))
import torch, numpy as np
import vtseq as V
if DEVICE == "cuda" and not torch.cuda.is_available(): DEVICE = "cpu"
print("torch", torch.__version__, "| device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
data, vocab = V.load_data(REPO)
print({s: len(v) for s, v in data.items()})

In [ ]:
# unit test cua moc 1 (CPU, ~10 s) - moi test phai PASS truoc khi chay
r = subprocess.run([sys.executable, "src/models/test_vtseq.py"], capture_output=True, text=True)
print(r.stdout[-2500:]); assert r.returncode == 0, r.stderr[-2000:]

## 5. Moc 2 - chuoi ablation M1 -> M4(b): OOF tren Train (CV 5 fold theo nhom)

In [ ]:
OUT = os.path.join(REPO, "outputs")
recs = []
for name in (CONFIGS if RUN_M2 else []):
    cfg = V.PRESETS[name]
    for seed in SEEDS:
        t0 = time.time()
        rec = V.run_oof_experiment(cfg, data, vocab, seed, OUT, root=REPO, device=DEVICE)
        recs.append(rec); o = rec["oof"]
        print(f"{name} s{seed}: macroF1_TTP {o['macro_f1_ttp']:.3f} | PR-AUC {o['pr_auc_bin']:.3f} | "
              f"FPR_HN {o.get('fpr_hard_negative', float('nan')):.3f} | epoch cuoi {rec['final_epochs']} | "
              f"fold cham tran {rec['n_hit_ceiling']} | {time.time() - t0:.0f}s", flush=True)

## 6. Bang OOF (trung binh +- do lech chuan qua seed) - tieu chi chon 4.3: macro-F1 TTP, PR-AUC, FPR hard_negative

In [ ]:
from collections import defaultdict
assert RUN_M2 or not recs
agg = defaultdict(lambda: defaultdict(list))
for r in recs:
    o = r["oof"]; a = agg[r["config"]["name"]]
    for k in ("macro_f1_ttp", "pr_auc_bin", "fpr_hard_negative", "fpr_neg", "recall"): a[k].append(o.get(k, np.nan))
    for t, v in o["f1_ttp"].items(): a["f1_" + t].append(v)
    a["n_hit_ceiling"].append(r["n_hit_ceiling"]); a["binary_score"].append(r["selection"]["binary_score"])
cols = ["macro_f1_ttp", "pr_auc_bin", "fpr_hard_negative", "f1_T1654", "n_hit_ceiling"]
print("| cau hinh | " + " | ".join(cols) + " | p_bin chon |")
for name, a in agg.items():
    cells = [f"{np.mean(a[c]):.3f} +- {np.std(a[c], ddof=1) if len(a[c]) > 1 else 0:.3f}" for c in cols]
    print(f"| {name} | " + " | ".join(cells) + f" | {dict((x, a['binary_score'].count(x)) for x in set(a['binary_score']))} |")

## 7. Dong goi ket qua (commit + checksum) len Drive -> tai ve host

In [ ]:
import hashlib, shutil
if RUN_M2:
  man = json.load(open(os.path.join(REPO, "reports", "step6", "aux_manifest.json")))
  ds = json.load(open(os.path.join(REPO, "reports", "dataset_stats.json")))["files_sha256"]
  info = {"commit": head, "torch": torch.__version__, "device": DEVICE, "configs": CONFIGS, "seeds": SEEDS,
          "dataset_sha256": {k: v for k, v in ds.items() if k.startswith("splits/") or k == "vocab.json"},
          "aux_sha256": man["sha256"], "n_runs": len(recs)}
  json.dump(info, open(os.path.join(OUT, "run_info.json"), "w"), indent=1)
  zp = shutil.make_archive(os.path.join(DRIVE_DIR, f"step6_m2_{head[:8]}"), "zip", OUT)
  print(zp, hashlib.sha256(open(zp, "rb").read()).hexdigest()[:16])

## Moc 3 - VT-Seq cuoi (M3, KHOA o reports/step6/final_config.json) + ablation tren test

Quy tac: cau hinh, nguong, p_bin/p_bin', so epoch cuoi deu lay tu OOF Train (khong chon lai gi tren test). Moi cau hinh x seed:
fit lai toan Train (so epoch = trung vi fold cua OOF), du doan t_clean / t_hard / t_novel MOT lan, + diem phat hien som.
- M1, M2, M3, M4, M4b x 5 seed: OOF + nguong lay tu moc 2 (`reports/step6/m2`, trong repo). Chi de BAO ablation tren test.
- M3c / M3cp: ablation (c) / (c') cho M3 - mat na ap ca khi train va khi test; chua co OOF nen chay CV 5 fold o day.
- VT-Seq-ens: trung binh xac suat 5 seed M3 (va M4b); nguong chon lai tren OOF trung binh.
- KHONG tinh chi so o day: tai zip ve, cham bang `src/eval/evaluate.py` tren host.

In [ ]:
from dataclasses import asdict
fc = json.load(open(os.path.join(REPO, "reports", "step6", "final_config.json")))
assert fc["config"] == asdict(V.PRESETS["M3"]) and fc["seeds"] == SEEDS, "cau hinh M3 khac ban da KHOA"
M2_DIR = os.path.join(REPO, "reports", "step6", "m2")
OUT3 = os.path.join(REPO, "outputs3")
FINAL_CONFIGS = ["M1", "M2", "M3", "M4", "M4b", "M3c", "M3cp"]
print("khoa:", fc["final_model"], "| locked_from_commit", fc["locked_from_commit"][:8])

In [ ]:
recs3 = []
for name in (FINAL_CONFIGS if RUN_M3 else []):
    for seed in SEEDS:
        r = V.run_final(V.PRESETS[name], data, vocab, seed, OUT3, root=REPO, device=DEVICE, oof_dir=M2_DIR)
        recs3.append(r)
        print(f"{r['model']}: epoch cuoi {r['final_epochs']} | {r['binary_score']} thr {r['thr']:.3f} | OOF tu {os.path.basename(str(r['oof_source']))} | {r['minutes']} ph", flush=True)

In [ ]:
ens = [V.build_ensemble(c, SEEDS, OUT3, data, root=REPO) for c in (("M3", "M4b") if RUN_M3 else ())]
for e in ens: print(e)

In [ ]:
import hashlib, shutil
if RUN_M3:
  man = json.load(open(os.path.join(REPO, "reports", "step6", "aux_manifest.json")))
  ds = json.load(open(os.path.join(REPO, "reports", "dataset_stats.json")))["files_sha256"]
  info = {"commit": head, "torch": torch.__version__, "device": DEVICE, "configs": FINAL_CONFIGS, "seeds": SEEDS,
          "final_config_locked_from": fc["locked_from_commit"],
          "dataset_sha256": {k: v for k, v in ds.items() if k.startswith("splits/") or k == "vocab.json"},
          "aux_sha256": man["sha256"], "runs": recs3, "ensembles": ens}
  json.dump(info, open(os.path.join(OUT3, "run_info.json"), "w"), indent=1)
  zp = shutil.make_archive(os.path.join(DRIVE_DIR, f"step6_m3_{head[:8]}"), "zip", OUT3)
  print(zp, hashlib.sha256(open(zp, "rb").read()).hexdigest()[:16])